In [2]:
import numpy as np
import os
import time
from scipy.io import loadmat, savemat

# -----------------------------
# Parameters
# -----------------------------
number = 200
nsx = 32
nsy = 32
nbz = 16
density = 1000  # kg/m^3 (1 g/cm^3)
# generate tra&val data
# generate syn data
path_m = './model/syn/'
path_d = './data/syn/'

# -----------------------------
# Load Txz matrix
# -----------------------------
print("Loading Tyy matrix ...")
Tyy_data = loadmat('./Tyy_matrix.mat')
Tyy = Tyy_data['Tyy']
print(f"Tyy matrix loaded: shape = {Tyy.shape}")

start_time = time.time()

for num in range(number):
    file_path = os.path.join(path_m, f'model{num}.dat')
    save_path = os.path.join(path_d, f'data{num}.mat')

    # Skip if file not found
    if not os.path.exists(file_path):
        print(f"⚠️ Warning: file not found {file_path}, skipping.")
        continue

    # Load model file
    input_data = np.loadtxt(file_path)
    if input_data.ndim == 1:
        input_data = input_data.reshape(1, -1)

    modeling_m, modeling_n = input_data.shape

    # Initialize model vector
    m = np.zeros((nsx * nsy * nbz, 1))

    # Fill model
    for i in range(modeling_m):
        ix = int(input_data[i, 0])
        iy = int(input_data[i, 1])
        iz = int(input_data[i, 2])
        val = input_data[i, 3]

        index = (ix-1) + nsx * (iy - 1) + nsx * nsy * (iz - 1)
        m[index] = val * density

    # Compute gravity response
    d = Tyy @ m

    # Save to .mat
    savemat(save_path, {'d': d, 'm': m})

    if num % 100 == 0:
        print(f"Processed model {num}/{number}")

end_time = time.time()
print(f"\n✅ Completed {number} models in {end_time - start_time:.2f} seconds")


Loading Tyy matrix ...
Tyy matrix loaded: shape = (1024, 16384)
Processed model 0/200
Processed model 100/200

✅ Completed 200 models in 2.44 seconds
